In [ ]:
import json

with open("<TODO>/basque_linguistic_competence/logs/data-all.json", "r", encoding="utf-8") as f:
    data = json.loads(f.read())

del f

In [ ]:
import numpy as np
from scipy import stats

def item_scores(models, section, task):
    per_model = []
    for m in models:
        assert "per_pair" in data[m] and task in data[m]["per_pair"].get(section, {}), f"No per-item predictions for {m} {section} {task}: rerun saving per-item outputs."
        prompts = data[m]["per_pair"][section][task]
        arr = np.array([[np.nan if x is None else x for x in v] for v in prompts.values()], dtype=float)
        per_model.append(arr.mean(axis=0))
    return np.mean(per_model, axis=0)

def paired_item_test(a, b):
    mask = ~np.isnan(a) & ~np.isnan(b)
    a, b = a[mask], b[mask]
    if np.allclose(a - b, 0.0):
        p_value = 1.0
    else:
        _, p_value = stats.ttest_rel(a, b)
    return a.mean(), b.mean(), p_value, int(mask.sum())

def holm(p_values):
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    adjusted = np.empty(m)
    running = 0.0
    for rank, idx in enumerate(np.argsort(p)):
        running = max(running, (m - rank) * p[idx])
        adjusted[idx] = min(1.0, running)
    return adjusted

def angle_symbol(mean_diff, p_value):
    if p_value > 0.05:
        return "\\sim"
    elif 0.01 < p_value <= 0.05:
        return "\\subtleup" if - mean_diff > 0 else "\\subtledown"
    else:
        return "\\nearrow" if - mean_diff > 0 else "\\searrow"

def print_table(rows, size="scriptsize"):
    p_holm = holm([r[3] for r in rows])
    for (name, m_a, m_b, p_value, n), p_adj in zip(rows, p_holm):
        mean_diff = m_a - m_b
        angle = angle_symbol(mean_diff, p_adj)
        print(f"{name} & {m_a * 100:.2f} \\({angle}\\) {m_b * 100:.2f} \\,\\textcolor{{{"darkred" if mean_diff > 0.0 else "darkgreen"}}}{{\\{size} ({- mean_diff * 100:+.2f})}} \\\\")

In [ ]:
families = [
    ("Apertus", ["swiss-ai/Apertus-8B-2509", "swiss-ai/Apertus-70B-2509"], ["swiss-ai/Apertus-8B-Instruct-2509", "swiss-ai/Apertus-70B-Instruct-2509"]),
    ("Gemma-3", ["google/gemma-3-12b-pt", "google/gemma-3-27b-pt"], ["google/gemma-3-12b-it", "google/gemma-3-27b-it"]),
    ("Gemma-4", ["google/gemma-4-12B", "google/gemma-4-26B-A4B", "google/gemma-4-31B"], ["google/gemma-4-12B-it", "google/gemma-4-26B-A4B-it", "google/gemma-4-31B-it"]),
    ("GLM-4", ["zai-org/GLM-4-32B-Base-0414", "zai-org/GLM-4.5-Air-Base"], ["zai-org/GLM-4-32B-0414", "zai-org/GLM-4.5-Air"]),
    ("LLaMA-3", ["meta-llama/Llama-3.1-8B", "meta-llama/Llama-3.1-70B"], ["meta-llama/Llama-3.1-8B-Instruct", "meta-llama/Llama-3.1-70B-Instruct"]),
    ("LLaMA-4", ["meta-llama/Llama-4-Scout-17B-16E"], ["meta-llama/Llama-4-Scout-17B-16E-Instruct"]),
    ("Qwen-3.5", ["Qwen/Qwen3.5-9B-Base", "Qwen/Qwen3.5-35B-A3B-Base"], ["Qwen/Qwen3.5-9B", "Qwen/Qwen3.5-35B-A3B"]),
    ("Salamandra", ["BSC-LT/salamandra-7b", "BSC-LT/ALIA-40b"], ["BSC-LT/salamandra-7b-instruct", "BSC-LT/ALIA-40b-instruct-2605"]),
    ("Kimu", ["orai-nlp/Gemma-Kimu-9b-base", "orai-nlp/Llama-eus-8B"], ["orai-nlp/Gemma-Kimu-9b-it", "orai-nlp/Llama-eus-8B-DIPLomA"]),
    ("Latxa", ["HiTZ/Latxa-Llama-3.1-8B"], ["HiTZ/Latxa-Llama-3.1-8B-Instruct"]),
]

rows = []
for name, base_models, instruct_models in families:
    b = item_scores(base_models, "LOGPROBS", "bat-ctxt")
    i = item_scores(instruct_models, "LOGPROBS", "bat-inst")
    rows.append((name, *paired_item_test(b, i)))

print_table(rows, size="footnotesize")

del families, rows, name, base_models, instruct_models, b, i

In [ ]:
families = [
    ("Apertus", ["swiss-ai/Apertus-8B-2509", "swiss-ai/Apertus-70B-2509"]),
    ("Gemma-3", ["google/gemma-3-12b-pt", "google/gemma-3-27b-pt"]),
    ("Gemma-4", ["google/gemma-4-12B", "google/gemma-4-26B-A4B", "google/gemma-4-31B"]),
    ("GLM-4", ["zai-org/GLM-4-32B-Base-0414", "zai-org/GLM-4.5-Air-Base"]),
    ("LLaMA-3", ["meta-llama/Llama-3.1-8B", "meta-llama/Llama-3.1-70B"]),
    ("LLaMA-4", ["meta-llama/Llama-4-Scout-17B-16E"]),
    ("Qwen-3.5", ["Qwen/Qwen3.5-9B-Base", "Qwen/Qwen3.5-35B-A3B-Base"]),
    ("Salamandra", ["BSC-LT/salamandra-7b", "BSC-LT/ALIA-40b"]),
    ("Kimu", ["orai-nlp/Gemma-Kimu-9b-base", "orai-nlp/Llama-eus-8B"]),
    ("Latxa", ["HiTZ/Latxa-Llama-3.1-8B"]),
]

rows = []
for name, models in families:
    s = item_scores(models, "LOGPROBS", "bat-sent")
    c = item_scores(models, "LOGPROBS", "bat-ctxt")
    rows.append((name, *paired_item_test(s, c)))

print_table(rows, size="scriptsize")

del families, rows, name, models, s, c

In [ ]:
def open_model(name, model):
    return (name, ([model], "BENCHMARK", "ab-instruct_BAT"), ([model], "THINKING", "ab-instruct_BAT-THINKING-SAMPLING"))

def gpt_model(name, no_reasoning, reasoning):
    return (name, ([no_reasoning], "BENCHMARK", "ab-instruct_BAT"), ([reasoning], "BENCHMARK", "ab-instruct_BAT"))

models = [
    open_model("Gemma-4-12B", "google/gemma-4-12B-it"),
    open_model("Gemma-4-26B-A4B", "google/gemma-4-26B-A4B-it"),
    open_model("Gemma-4-31B", "google/gemma-4-31B-it"),
    open_model("GLM-4.5", "zai-org/GLM-4.5-Air"),
    open_model("GLM-4.6V", "zai-org/GLM-4.6V"),
    open_model("GLM-4.6V-Flash", "zai-org/GLM-4.6V-Flash"),
    open_model("Qwen3.5-9B", "Qwen/Qwen3.5-9B"),
    open_model("Qwen3.5-27B", "Qwen/Qwen3.5-27B"),
    open_model("Qwen3.5-35B-A3B", "Qwen/Qwen3.5-35B-A3B"),
    open_model("Qwen3.5-122B-A10B", "Qwen/Qwen3.5-122B-A10B"),
    open_model("Qwen3.6-27B", "Qwen/Qwen3.6-27B"),
    open_model("Qwen3.6-35B-A3B", "Qwen/Qwen3.6-35B-A3B"),
    gpt_model("gpt-5-2025-08-07", "gpt-5-2025-08-07_minimal", "gpt-5-2025-08-07_high"),
    gpt_model("gpt-5.1-2025-11-13", "gpt-5.1-2025-11-13_none", "gpt-5.1-2025-11-13_high"),
]

rows = []
for name, no_reasoning, reasoning in models:
    i = item_scores(*no_reasoning)
    t = item_scores(*reasoning)
    rows.append((name, *paired_item_test(i, t)))

print_table(rows, size="scriptsize")

del open_model, gpt_model, models, rows, name, no_reasoning, reasoning, i, t